# Nulling CDF Simulation


In [ ]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import chi2
from sionna.rt import load_scene

import SceneConfigSionna2
import nulling_cdf_utils as ncu

importlib.reload(SceneConfigSionna2)
importlib.reload(ncu)

from SceneConfigSionna2 import SceneConfigSionna
from nulling_cdf_utils import (
    run_nulling_cdf_experiment,
    save_experiment_metrics,
)



In [ ]:
scene = load_scene("blender_scene_big/10km_times_10km/10km_times_10km.xml")
SceneConfig = SceneConfigSionna(scene)
SceneConfig.build_coverage_map(grid_size=10, show_xy=True, plot=False)


## Simulation Parameters


In [ ]:
# Geometry and deployment
ntn_rx = 200
tn_rx = 300
bs_row = 3
bs_col = 3
nbs = bs_row * bs_col
nsect = 3

# Satellite direction used by compute_positions in every macro simulation
azimuth = float(np.random.uniform(0.0, 360.0))
elevation = float(np.random.uniform(35.0, 90.0))

# Carrier and array configuration
fc = 9.99e9
tx_antenna_rows = 8
tx_antenna_cols = 8
tn_rx_antenna_rows = 1
tn_rx_antenna_cols = 1
tx_antennas = tx_antenna_rows * tx_antenna_cols

# TX sector orientation
# Sionna orientation order: [yaw, pitch, roll]
tx_sector_yaw_offset_deg = 0.0
tx_head_down_deg = 5.0
tx_sector_roll_deg = 0.0

tx_sector_yaw_offset_rad = np.deg2rad(tx_sector_yaw_offset_deg)
tx_sector_pitch_rad = -np.deg2rad(tx_head_down_deg)
tx_sector_roll_rad = np.deg2rad(tx_sector_roll_deg)

# Monte Carlo setting
num_macro_sims = 2
show_progress = True
plot_layout_on_first_sim = False

print(
    f"TX sector orientation: yaw_offset={tx_sector_yaw_offset_deg:.1f} deg, "
    f"head_down={tx_head_down_deg:.1f} deg, roll={tx_sector_roll_deg:.1f} deg"
)
print(f"Satellite azimuth={azimuth:.2f} deg, elevation={elevation:.2f} deg")


In [ ]:
# Noise, thresholds, and MUSIC setup
EkT = -174
B = 100e6
Tx_power_dbm = 30
Tx_power = 10 ** ((Tx_power_dbm - 30) / 10)
Tx_power_handheld_dbm = 23
Tx_power_handheld = 10 ** ((Tx_power_handheld_dbm - 30) / 10)

NF = 7
NF_vsat = 3
NF_bs = 2
N0_dBm = EkT + 10 * np.log10(B) + NF
N0 = 10 ** ((N0_dBm - 30) / 10)
N0_vsat = 10 ** ((EkT + 10 * np.log10(B) + NF_vsat - 30) / 10)
N0_bs = 10 ** ((EkT + 10 * np.log10(B) + NF_bs - 30) / 10)

preamble_time = 20e-6
N0_sigma = N0_vsat / Tx_power / preamble_time
N0_sigma_handheld = 10 ** ((EkT + NF_bs - 30) / 10) / Tx_power_handheld / preamble_time

lambda_ranges = [0, 1e10, 1e11, 1e12]

snr_threshold = -6
inr_threshold = -6
h_ntn_th = np.sqrt(10 ** (inr_threshold / 10) * N0_bs * tx_antennas / Tx_power)
h_tn_th = np.sqrt(10 ** (snr_threshold / 10) * N0_bs * tx_antennas / Tx_power)
threshold_peft_db = 10 * np.log10(np.abs(h_tn_th) ** 2)
p_fa = 1 / B
pfa_threshold = chi2.ppf(1 - p_fa, 2 * tx_antennas) / 2
h_ntn_pfa_th = np.sqrt(pfa_threshold * N0_sigma)
h_tn_pfa_th = np.sqrt(pfa_threshold * N0_sigma_handheld)
threshold_ntn_pfa_db = 10 * np.log10(np.abs(h_ntn_pfa_th) ** 2)
threshold_tn_pfa_db = 10 * np.log10(np.abs(h_tn_pfa_th) ** 2)

sionna_phi_is_global = True
theta_display_mode = "elevation"

music_num_sources = None
music_threshold = 3.0
music_covariance_mode = "sample"
music_num_snapshots = 100
music_noise_var = N0_bs / Tx_power
music_rng_seed = 7
music_source_estimation = "mdl"
music_energy_ratio = 0.95
music_reduce_ntn_ant = "max"
music_user_powers = None
music_use_sector_orientation = True
music_sector_pitch_rad = float(tx_sector_pitch_rad)
music_sector_roll_rad = float(tx_sector_roll_rad)
music_rotation_order = "zyx"
music_std_channel_mode = "conj"
music_std_manifold_label = "yz:+1"
music_std_flatten_order = "F"
music_std_scan_mode = "complex"
music_std_phi_offset_deg = 0.0
music_std_phi_mirror_about_sector = False
music_std_horizontal_sign = -1
music_sector_forward_only = True
music_sector_forward_cos_min = 0.0
music_phi_grid_deg = np.arange(0.0, 360.0, 1.0)
music_theta_grid_deg = np.arange(0.0, 181.0, 1.0)

print(f"h_tn_th = {h_tn_th:.4e}")
print(f"h_ntn_th = {h_ntn_th:.4e}")
print(f"TN Pfa threshold = {threshold_tn_pfa_db:.2f} dB")
print(f"NTN Pfa threshold = {threshold_ntn_pfa_db:.2f} dB")
print(f"Beamforming threshold = {threshold_peft_db:.2f} dB")



## Monte Carlo Experiment

- Each macro simulation calls `compute_positions(...)` and `compute_paths(...)`.
- BS locations remain fixed because the same grid deployment is used in every macro simulation.
- TN users are paired to the strongest valid TX over all BS sectors.
- If the minimum number of paired TN users across all TX sectors is `m`, each macro simulation runs exactly `m` small rounds.
- In each small round, every TX serves one paired TN and nulls all NTN users detected by MUSIC for that TX.
- INR is computed as the sum of interference powers from all TX beams over the noise power.


In [ ]:
result_dir = Path("result")
result_dir.mkdir(parents=True, exist_ok=True)

compute_positions_kwargs_mc = dict(
    ntn_rx=ntn_rx,
    tn_rx=tn_rx,
    azimuth=azimuth,
    elevation=elevation,
    centerBS=False,
    bs_grid=(bs_row, bs_col),
    bs_boundary=1500,
    tn_building_ratio=0.6,
    tn_distance=400,
    ntn_building_ratio=0.6,
    plot_grid=plot_layout_on_first_sim,
    plot_bs=plot_layout_on_first_sim,
    plot_tn=plot_layout_on_first_sim,
    plot_ntn=plot_layout_on_first_sim,
)

compute_paths_kwargs_mc = dict(
    nsect=nsect,
    fc=fc,
    tx_rows=tx_antenna_rows,
    tx_cols=tx_antenna_cols,
    tn_rx_rows=tn_rx_antenna_rows,
    tn_rx_cols=tn_rx_antenna_cols,
    max_depth=0,
    bandwidth=B,
    tx_power_dbm=Tx_power_dbm,
    sector_yaw_offset_rad=tx_sector_yaw_offset_rad,
    sector_pitch_rad=tx_sector_pitch_rad,
    sector_roll_rad=tx_sector_roll_rad,
)

music_kwargs_mc = dict(
    tx_rows=int(tx_antenna_rows),
    tx_cols=int(tx_antenna_cols),
    nsect=int(nsect),
    pair_keys=None,
    detect_num_sources=music_num_sources,
    detect_threshold=music_threshold,
    detect_user_powers=music_user_powers,
    detect_noise_var=music_noise_var,
    detect_covariance_mode=music_covariance_mode,
    detect_num_snapshots=music_num_snapshots,
    detect_rng_seed=music_rng_seed,
    detect_source_estimation=music_source_estimation,
    detect_energy_ratio=music_energy_ratio,
    detect_reduce_rx_ant=music_reduce_ntn_ant,
    channel_mode=music_std_channel_mode,
    manifold_label=music_std_manifold_label,
    flatten_order=music_std_flatten_order,
    scan_mode=music_std_scan_mode,
    phi_offset_deg=music_std_phi_offset_deg,
    phi_mirror_about_sector=music_std_phi_mirror_about_sector,
    steering_horizontal_sign=music_std_horizontal_sign,
    use_sector_orientation=music_use_sector_orientation,
    sector_pitch_rad=music_sector_pitch_rad,
    sector_roll_rad=music_sector_roll_rad,
    rotation_order=music_rotation_order,
    sector_forward_only=music_sector_forward_only,
    sector_forward_cos_min=music_sector_forward_cos_min,
    phi_grid_deg=music_phi_grid_deg,
    theta_grid_deg=music_theta_grid_deg,
)

nulling_cdf_results = run_nulling_cdf_experiment(
    SceneConfig,
    num_macro_sims=num_macro_sims,
    compute_positions_kwargs=compute_positions_kwargs_mc,
    compute_paths_kwargs=compute_paths_kwargs_mc,
    lambda_ranges=lambda_ranges,
    h_tn_th=h_tn_th,
    tx_antennas=tx_antennas,
    tx_power=Tx_power,
    noise_power=N0,
    music_kwargs=music_kwargs_mc,
    sionna_phi_is_global=sionna_phi_is_global,
    theta_display_mode=theta_display_mode,
    plot_first_sim_only=plot_layout_on_first_sim,
    show_progress=show_progress,
)

metrics_path = save_experiment_metrics(
    nulling_cdf_results,
    result_dir=result_dir,
    output_name="nulling_cdf_metrics.npz",
)

macro_stats = nulling_cdf_results["macro_stats"]
macro_min_count = np.asarray([row["min_count"] for row in macro_stats], dtype=int)
macro_detected_ntn = np.asarray([row["detected_ntn_count"] for row in macro_stats], dtype=int)
macro_interfered_ntn = np.asarray([row["interfered_ntn_count"] for row in macro_stats], dtype=int)
macro_phi_mae_deg = np.asarray([row["angle_metrics"].get("phi_mae_deg", np.nan) for row in macro_stats], dtype=np.float64)
macro_elev_mae_deg = np.asarray([row["angle_metrics"].get("elev_mae_deg", np.nan) for row in macro_stats], dtype=np.float64)
macro_angle_match_count = np.asarray([row["angle_metrics"].get("matched_pairs", 0) for row in macro_stats], dtype=int)
macro_subset_count = np.asarray([row["detected_subset_metrics"].get("count", 0) for row in macro_stats], dtype=np.float64)
macro_subset_nrmse = np.asarray([row["detected_subset_metrics"].get("nrmse", np.nan) for row in macro_stats], dtype=np.float64)
macro_subset_cos_sim = np.asarray([row["detected_subset_metrics"].get("cos_sim", np.nan) for row in macro_stats], dtype=np.float64)
macro_subset_mag_mae = np.asarray([row["detected_subset_metrics"].get("mag_mae", np.nan) for row in macro_stats], dtype=np.float64)
macro_subset_power_ratio_db = np.asarray([row["detected_subset_metrics"].get("power_ratio_db", np.nan) for row in macro_stats], dtype=np.float64)
macro_pairs_count = np.asarray([row["detected_pairs_summary"].get("pairs", 0) for row in macro_stats], dtype=int)
macro_pairs_nrmse_mean = np.asarray([row["detected_pairs_summary"].get("nrmse_mean", np.nan) for row in macro_stats], dtype=np.float64)
macro_pairs_nrmse_median = np.asarray([row["detected_pairs_summary"].get("nrmse_median", np.nan) for row in macro_stats], dtype=np.float64)
macro_pairs_cos_mean = np.asarray([row["detected_pairs_summary"].get("cos_mean", np.nan) for row in macro_stats], dtype=np.float64)
macro_pairs_cos_median = np.asarray([row["detected_pairs_summary"].get("cos_median", np.nan) for row in macro_stats], dtype=np.float64)

print(f"Number of macro simulations: {num_macro_sims}")
print(f"Macro simulations with min_count > 0: {np.count_nonzero(macro_min_count > 0)}")
print(f"Average min_count: {macro_min_count.mean() if macro_min_count.size else 0:.2f}")
print(f"Average detected NTN count: {macro_detected_ntn.mean() if macro_detected_ntn.size else 0:.2f}")
print(f"Average interfered NTN count: {macro_interfered_ntn.mean() if macro_interfered_ntn.size else 0:.2f}")
print(f"Average phi MAE: {np.nanmean(macro_phi_mae_deg) if macro_phi_mae_deg.size else np.nan:.3f} deg")
print(f"Average elevation MAE: {np.nanmean(macro_elev_mae_deg) if macro_elev_mae_deg.size else np.nan:.3f} deg")
print(f"Raw INR samples: {nulling_cdf_results['raw_inr_db'].size}")
print(f"Raw SNR samples: {nulling_cdf_results['raw_snr_db'].size}")
for lambda_ in lambda_ranges:
    print(
        f"lambda={lambda_:.0e}: "
        f"INR samples={nulling_cdf_results['null_inr_db'][float(lambda_)].size}, "
        f"SNR samples={nulling_cdf_results['null_snr_db'][float(lambda_)].size}"
    )
print(f"Metrics saved to: {metrics_path}")

header = (
    f"{'sim':>3} {'m':>3} {'detNTN':>6} {'pathNTN':>7} {'angN':>5} {'phiMAE':>8} {'elevMAE':>8} "
    f"{'subN':>8} {'subNRMSE':>10} {'subCos':>8} {'sub|h|MAE':>10} {'subPR':>8} "
    f"{'pairN':>6} {'pairNRMSEm':>11} {'pairNRMSEd':>11} {'pairCosM':>9} {'pairCosD':>9}"
)
print("
Per-macro MUSIC summary")
print(header)
print("-" * len(header))
for idx, row in enumerate(macro_stats):
    print(
        f"{idx:3d} "
        f"{macro_min_count[idx]:3d} "
        f"{macro_detected_ntn[idx]:6d} "
        f"{macro_interfered_ntn[idx]:7d} "
        f"{macro_angle_match_count[idx]:5d} "
        f"{macro_phi_mae_deg[idx]:8.3f} "
        f"{macro_elev_mae_deg[idx]:8.3f} "
        f"{macro_subset_count[idx]:8.0f} "
        f"{macro_subset_nrmse[idx]:10.4e} "
        f"{macro_subset_cos_sim[idx]:8.4f} "
        f"{macro_subset_mag_mae[idx]:10.4e} "
        f"{macro_subset_power_ratio_db[idx]:8.3f} "
        f"{macro_pairs_count[idx]:6d} "
        f"{macro_pairs_nrmse_mean[idx]:11.4e} "
        f"{macro_pairs_nrmse_median[idx]:11.4e} "
        f"{macro_pairs_cos_mean[idx]:9.4f} "
        f"{macro_pairs_cos_median[idx]:9.4f}"
    )



In [ ]:
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "lines.linewidth": 1.4,
    }
)

fig, ax = plt.subplots(figsize=(3.5, 2.6))

raw_inr = np.asarray(nulling_cdf_results["raw_inr_db"], dtype=np.float64)
raw_inr = raw_inr[np.isfinite(raw_inr)]
if raw_inr.size > 0:
    raw_inr_sorted = np.sort(raw_inr)
    raw_inr_cdf = np.arange(1, raw_inr_sorted.size + 1, dtype=np.float64) / float(raw_inr_sorted.size)
    ax.plot(raw_inr_sorted, raw_inr_cdf, color="black", linestyle="-", label="No nulling")

color_cycle = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#8c564b", "#17becf"]
style_cycle = ["--", "-.", ":", (0, (4, 1, 1, 1)), (0, (6, 2)), (0, (3, 1, 1, 1, 1, 1))]

for idx, lambda_ in enumerate(sorted(nulling_cdf_results["null_inr_db"].keys())):
    inr_values = np.asarray(nulling_cdf_results["null_inr_db"][lambda_], dtype=np.float64)
    inr_values = inr_values[np.isfinite(inr_values)]
    if inr_values.size == 0:
        continue

    inr_sorted = np.sort(inr_values)
    inr_cdf = np.arange(1, inr_sorted.size + 1, dtype=np.float64) / float(inr_sorted.size)
    ax.plot(
        inr_sorted,
        inr_cdf,
        color=color_cycle[idx % len(color_cycle)],
        linestyle=style_cycle[idx % len(style_cycle)],
        label=f"$\lambda={float(lambda_):.0e}$",
    )

ax.set_xlabel("INR (dB)")
ax.set_ylabel("CDF")
ax.set_title("CDF of INR with and without Nulling")
ax.set_ylim(0.0, 1.0)
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.8)
ax.legend(loc="lower right", frameon=True)
fig.tight_layout(pad=0.2)

inr_png_path = result_dir / "nulling_inr_cdf.png"
inr_pdf_path = result_dir / "nulling_inr_cdf.pdf"
fig.savefig(inr_png_path, dpi=400, bbox_inches="tight")
fig.savefig(inr_pdf_path, bbox_inches="tight")
plt.show()

print(f"INR CDF PNG: {inr_png_path}")
print(f"INR CDF PDF: {inr_pdf_path}")



In [ ]:
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "font.size": 8,
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "legend.fontsize": 7,
        "xtick.labelsize": 7,
        "ytick.labelsize": 7,
        "lines.linewidth": 1.4,
    }
)

fig, ax = plt.subplots(figsize=(3.5, 2.6))

raw_snr = np.asarray(nulling_cdf_results["raw_snr_db"], dtype=np.float64)
raw_snr = raw_snr[np.isfinite(raw_snr)]
if raw_snr.size > 0:
    raw_snr_sorted = np.sort(raw_snr)
    raw_snr_cdf = np.arange(1, raw_snr_sorted.size + 1, dtype=np.float64) / float(raw_snr_sorted.size)
    ax.plot(raw_snr_sorted, raw_snr_cdf, color="black", linestyle="-", label="No nulling")

color_cycle = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#8c564b", "#17becf"]
style_cycle = ["--", "-.", ":", (0, (4, 1, 1, 1)), (0, (6, 2)), (0, (3, 1, 1, 1, 1, 1))]

for idx, lambda_ in enumerate(sorted(nulling_cdf_results["null_snr_db"].keys())):
    snr_values = np.asarray(nulling_cdf_results["null_snr_db"][lambda_], dtype=np.float64)
    snr_values = snr_values[np.isfinite(snr_values)]
    if snr_values.size == 0:
        continue

    snr_sorted = np.sort(snr_values)
    snr_cdf = np.arange(1, snr_sorted.size + 1, dtype=np.float64) / float(snr_sorted.size)
    ax.plot(
        snr_sorted,
        snr_cdf,
        color=color_cycle[idx % len(color_cycle)],
        linestyle=style_cycle[idx % len(style_cycle)],
        label=f"$\lambda={float(lambda_):.0e}$",
    )

ax.set_xlabel("SNR (dB)")
ax.set_ylabel("CDF")
ax.set_title("CDF of SNR with and without Nulling")
ax.set_ylim(0.0, 1.0)
ax.grid(True, linestyle=":", linewidth=0.6, alpha=0.8)
ax.legend(loc="lower right", frameon=True)
fig.tight_layout(pad=0.2)

snr_png_path = result_dir / "nulling_snr_cdf.png"
snr_pdf_path = result_dir / "nulling_snr_cdf.pdf"
fig.savefig(snr_png_path, dpi=400, bbox_inches="tight")
fig.savefig(snr_pdf_path, bbox_inches="tight")
plt.show()

print(f"SNR CDF PNG: {snr_png_path}")
print(f"SNR CDF PDF: {snr_pdf_path}")



In [ ]:
print("INR aggregation rule: total interference power from all TX beams divided by noise power.")
print("Detected-subset tensor metrics compare h_hat_all against h_ntn_all on detected NTN users only.")
print("Detected pairs summary compares each detected (rx, tx, rx_ant) channel vector independently.")
print("NRMSE is normalized reconstruction error and lower is better.")
print("CosSim is normalized vector similarity and higher is better.")
print("PowerRatio is estimated power over true power in dB; 0 dB is ideal.")

